In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, models, transforms
from torchvision.transforms import functional as TF
from tqdm.auto import tqdm

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else
                      "mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", device)
print("Torch:", torch.__version__)

### Data Handling, Watermarking, ResNet50 Model Training, Model Evaluation

In [ ]:
DATA_ROOT = "./data"
MAX_TRAIN_IMAGES = None   # Example: 4000 for a quick dry run; None = full training data
MAX_TEST_IMAGES = None    # Example: 1000 for a quick dry run; None = full test data
VAL_FRACTION = 0.10
BATCH_SIZE = 128
NUM_WORKERS = 0
IMAGE_SIZE = 32

cifar_train = datasets.CIFAR10(DATA_ROOT, train=True, download=True)
cifar_test = datasets.CIFAR10(DATA_ROOT, train=False, download=True)

print("Raw CIFAR-10:", len(cifar_train), "train,", len(cifar_test), "test")
print("Classes:", cifar_train.classes)

# Stratify by CIFAR-10 object category before assigning watermark labels.
def stratified_indices(targets, selected_count=None, seed=42):
    targets = np.asarray(targets)
    rng = np.random.default_rng(seed)
    classes = np.unique(targets)
    if selected_count is None:
        selected_count = len(targets)
    if selected_count % (2 * len(classes)) != 0:
        raise ValueError("For balanced per-category watermark labels, selected_count must be divisible by 20.")
    per_class = selected_count // len(classes)
    indices = np.concatenate([
        rng.choice(np.where(targets == cls)[0], size=per_class, replace=False)
        for cls in classes
    ])
    return rng.permutation(indices)

all_train_indices = stratified_indices(cifar_train.targets, MAX_TRAIN_IMAGES, seed=SEED)
test_indices = stratified_indices(cifar_test.targets, MAX_TEST_IMAGES, seed=SEED + 1)

# Split within CIFAR object classes, so both train and validation stay stratified.
rng = np.random.default_rng(SEED + 2)
train_list, val_list = [], []
train_targets = np.asarray(cifar_train.targets)
for cls in np.unique(train_targets):
    class_indices = all_train_indices[train_targets[all_train_indices] == cls].copy()
    rng.shuffle(class_indices)
    n_val = round(len(class_indices) * VAL_FRACTION)
    if n_val % 2:  # keep watermark labels balanced within each CIFAR class
        n_val += 1
    val_list.extend(class_indices[:n_val].tolist())
    train_list.extend(class_indices[n_val:].tolist())
train_indices = rng.permutation(np.array(train_list, dtype=int))
val_indices = rng.permutation(np.array(val_list, dtype=int))

print(f"Unique originals: train={len(train_indices):,}; val={len(val_indices):,}; test={len(test_indices):,}")
assert not set(train_indices).intersection(val_indices)

In [ ]:
# show 3 images from the training set (put on slides to introduce the dataset, with the labels in the previous cell)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(6, 2))

for i, ax in enumerate(axes):
    idx = train_indices[i]
    image, label = cifar_train[idx]
    ax.imshow(image)
    ax.set_title(cifar_train.classes[label])
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
WATERMARK_STRENGTH = 0.02  # Main parameter: try 0.005, 0.01, 0.02, 0.05
WATERMARK_SEED = 2026
assert 0 <= WATERMARK_STRENGTH <= 1

pattern_generator = torch.Generator().manual_seed(WATERMARK_SEED)
watermark_pattern = (
    torch.randint(0, 2, (3, 32, 32), generator=pattern_generator, dtype=torch.int64)
    .float().mul(2).sub(1)
)

def add_watermark(image, strength=WATERMARK_STRENGTH, pattern=watermark_pattern):
    """Image: float Tensor [3,32,32] in [0,1]; returns watermarked image in [0,1]."""
    if image.shape != pattern.shape:
        raise ValueError(f"Expected image shape {tuple(pattern.shape)}, got {tuple(image.shape)}")
    return torch.clamp(image + float(strength) * pattern.to(image.device), 0, 1)

print("Watermark pattern shape:", tuple(watermark_pattern.shape))
print("Watermark strength:", WATERMARK_STRENGTH)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3.4))
pattern_rgb = watermark_pattern.permute(1, 2, 0).numpy()
axes[0].imshow((pattern_rgb + 1) / 2)
axes[0].set_title("Fixed RGB pattern (visualized)")
for c, label in enumerate(["Red channel", "Green channel", "Blue channel"]):
    axes[c + 1].imshow(watermark_pattern[c], cmap="gray", vmin=-1, vmax=1)
    axes[c + 1].set_title(label)
for ax in axes:
    ax.axis("off")
plt.suptitle(f"Watermark pattern | embedding strength λ={WATERMARK_STRENGTH}")
plt.tight_layout()
plt.show()

In [ ]:
to_tensor = transforms.ToTensor()
examples = [cifar_train[int(i)] for i in train_indices[:5]]
fig, axes = plt.subplots(len(examples), 3, figsize=(9, 3 * len(examples)))
for row, (pil_img, object_label) in enumerate(examples):
    clean = to_tensor(pil_img)
    marked = add_watermark(clean)
    diff = marked - clean
    axes[row, 0].imshow(clean.permute(1, 2, 0))
    axes[row, 1].imshow(marked.permute(1, 2, 0))
    # Amplified difference is ONLY for visualization; not the actual pixel strength.
    axes[row, 2].imshow(torch.clamp((diff / max(WATERMARK_STRENGTH, 1e-8) + 1) / 2, 0, 1).permute(1, 2, 0))
    axes[row, 0].set_ylabel(cifar_train.classes[object_label])
for ax, label in zip(axes[0], ["Original (clean)", "Watermarked", "Difference (amplified)"]):
    ax.set_title(label)
for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

# Image-quality baseline on examples, using unnormalized pixel tensors in [0,1].
mse_values = [torch.mean((add_watermark(to_tensor(im)) - to_tensor(im)) ** 2).item()
              for im, _ in examples]
psnr_values = [float("inf") if mse == 0 else 10 * np.log10(1.0 / mse) for mse in mse_values]
print(f"Example average MSE: {np.mean(mse_values):.8f}")
print(f"Example average PSNR: {np.mean(psnr_values):.2f} dB")

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

# Inputs are PIL originals; watermark is injected before resizing and normalization.
class WatermarkCIFAR10(Dataset):
    def __init__(self, cifar_dataset, indices, strength=WATERMARK_STRENGTH,
                 pattern=watermark_pattern, seed=42, image_size=32):
        self.base = cifar_dataset
        self.indices = np.asarray(indices, dtype=int)
        self.strength = strength
        self.pattern = pattern
        self.image_size = image_size
        self.labels = np.zeros(len(self.indices), dtype=np.int64)
        object_labels = np.asarray(self.base.targets)[self.indices]
        rng = np.random.default_rng(seed)
        for cls in np.unique(object_labels):
            positions = np.flatnonzero(object_labels == cls)
            if len(positions) % 2:
                raise ValueError("Each source class must have even count for a 50/50 label split")
            rng.shuffle(positions)
            self.labels[positions[:len(positions) // 2]] = 1

        self.resize = transforms.Resize((image_size, image_size), antialias=True)
        self.normalize = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = int(self.indices[i])

        # Load directly from CIFAR-10's NumPy data
        image = self.base.data[idx]

        # HWC uint8 -> CHW float32 in [0, 1]
        x = (
            torch.from_numpy(image)
            .permute(2, 0, 1)
            .float()
            .div_(255.0)
        )

        y = int(self.labels[i])

        if y == 1:
            x = add_watermark(
                x,
                strength=self.strength,
                pattern=self.pattern
            )

        # Only resize when necessary
        if self.image_size != 32:
            x = self.resize(x)

        x = self.normalize(x)

        return x, y

trainset = WatermarkCIFAR10(cifar_train, train_indices, seed=SEED + 10)
valset = WatermarkCIFAR10(cifar_train, val_indices, seed=SEED + 11)
testset = WatermarkCIFAR10(cifar_test, test_indices, seed=SEED + 12)

trainloader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True,
                         num_workers=NUM_WORKERS, pin_memory=device.type == "cuda")
valloader = DataLoader(valset, batch_size=BATCH_SIZE, shuffle=False,
                       num_workers=NUM_WORKERS, pin_memory=device.type == "cuda")
testloader = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=device.type == "cuda")

summary = pd.DataFrame([
    {"partition": name, "unwatermarked": int(np.sum(ds.labels == 0)),
     "watermarked": int(np.sum(ds.labels == 1)), "total": len(ds)}
    for name, ds in [("train", trainset), ("validation", valset), ("test", testset)]
])
display(summary)
assert all((row.unwatermarked == row.watermarked) for row in summary.itertuples())

In [ ]:
WEIGHTS = models.ResNet50_Weights.IMAGENET1K_V2
model = models.resnet50(weights=WEIGHTS)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
LEARNING_RATE = 0.001
MOMENTUM = 0.9
MAX_EPOCHS = 15
TOLERANCE = 1e-6

optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=MOMENTUM)
print("Model:", type(model).__name__)
print("Classification head:", model.fc)
print("Trainable parameters:", f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
@torch.no_grad()
def evaluate_model(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    all_targets = []
    all_preds = []
    all_prob_marked = []

    for images, targets in dataloader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        with torch.autocast(
            "cuda",
            enabled=device.type == "cuda"
        ):
            logits = model(images)
            loss = criterion(logits, targets)
        preds = logits.argmax(dim=1)
        total_loss += loss.item() * targets.size(0)
        correct += (preds == targets).sum().item()
        total += targets.size(0)
        all_targets.extend(targets.cpu().numpy().tolist())
        all_preds.extend(preds.cpu().numpy().tolist())
        all_prob_marked.extend(logits.softmax(dim=1)[:, 1].cpu().numpy().tolist())

    return {
        "loss": total_loss / total,
        "accuracy": correct / total,
        "targets": np.asarray(all_targets, dtype=int),
        "preds": np.asarray(all_preds, dtype=int),
        "p_watermarked": np.asarray(all_prob_marked, dtype=float),
    }


def train_torch_model_cifar10(model, trainloader, valloader, optimizer, criterion,
                              max_epochs, device, tolerance=1e-6):
    history = []
    prev_loss = float("inf")
    model.to(device)
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    for epoch in range(max_epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        last_batch_loss = None

        for images, targets in tqdm(trainloader, desc=f"Epoch {epoch+1}/{max_epochs}", leave=False):
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast("cuda", enabled=use_amp):
                logits = model(images)
                loss = criterion(logits, targets)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            last_batch_loss = loss.item()
            running_loss += last_batch_loss * targets.size(0)
            correct += (logits.argmax(dim=1) == targets).sum().item()
            total += targets.size(0)

        if last_batch_loss is None:
            raise ValueError("Training dataloader is empty.")

        val_results = evaluate_model(model, valloader, criterion, device)
        row = {
            "epoch": epoch + 1,
            "train_loss": running_loss / total,
            "last_batch_loss": last_batch_loss,
            "train_accuracy": correct / total,
            "val_loss": val_results["loss"],
            "val_accuracy": val_results["accuracy"],
        }
        history.append(row)
        print(f"Epoch {epoch+1:02d} | last batch loss {last_batch_loss:.6f} | "
              f"train loss {row['train_loss']:.4f} | "
              f"train acc {row['train_accuracy']:.2%} | "
              f"val loss {row['val_loss']:.4f} | "
              f"val acc {row['val_accuracy']:.2%}")

        if epoch > 0:
            loss_diff = prev_loss - last_batch_loss
            if loss_diff < tolerance:
                print(f"Early stopping at epoch {epoch}")
                break
        prev_loss = last_batch_loss

    return model, pd.DataFrame(history)

In [ ]:
# Full run: requires model download and potentially considerable GPU time.
# Start with MAX_TRAIN_IMAGES=4000 / MAX_TEST_IMAGES=1000 above for a quick smoke test.
model, history = train_torch_model_cifar10(
    model=model,
    trainloader=trainloader,
    valloader=valloader,
    optimizer=optimizer,
    criterion=criterion,
    max_epochs=MAX_EPOCHS,
    device=device,
    tolerance=TOLERANCE,
)

display(history.tail())

In [ ]:
def detection_metrics(results, split_name):
    y_true, y_pred = results["targets"], results["preds"]
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    safe = lambda numerator, denominator: numerator / denominator if denominator else float("nan")
    return {
        "split": split_name,
        "loss": results["loss"],
        "accuracy": results["accuracy"],
        "TPR / recall": safe(tp, tp + fn),
        "FPR": safe(fp, fp + tn),
        "precision": safe(tp, tp + fp),
        "TP": tp, "TN": tn, "FP": fp, "FN": fn,
    }

train_results = evaluate_model(model, trainloader, criterion, device)
val_results = evaluate_model(model, valloader, criterion, device)
test_results = evaluate_model(model, testloader, criterion, device)
metrics = pd.DataFrame([
    detection_metrics(train_results, "Train"),
    detection_metrics(val_results, "Validation"),
    detection_metrics(test_results, "Test"),
])
display(metrics.style.format({
    "loss": "{:.4f}", "accuracy": "{:.2%}", "TPR / recall": "{:.2%}",
    "FPR": "{:.2%}", "precision": "{:.2%}",
}))
print("TEST RESULTS")
print(f"Test accuracy: {metrics.iloc[-1]['accuracy']:.2%}")
print(f"Watermark detection rate (TPR): {metrics.iloc[-1]['TPR / recall']:.2%}")
print(f"False positive rate: {metrics.iloc[-1]['FPR']:.2%}")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
axs[0].plot(history["epoch"], history["train_loss"], marker="o", label="Train")
axs[0].plot(history["epoch"], history["val_loss"], marker="o", label="Validation")
axs[0].set(title="Cross-entropy loss", xlabel="Epoch", ylabel="Loss")
axs[0].legend()

axs[1].plot(history["epoch"], history["train_accuracy"], marker="o", label="Train")
axs[1].plot(history["epoch"], history["val_accuracy"], marker="o", label="Validation")
axs[1].set(title="Watermark classification accuracy", xlabel="Epoch", ylabel="Accuracy")
axs[1].legend()
plt.tight_layout()
plt.show()

cm = np.array([
    [int(((test_results["targets"] == 0) & (test_results["preds"] == 0)).sum()),
     int(((test_results["targets"] == 0) & (test_results["preds"] == 1)).sum())],
    [int(((test_results["targets"] == 1) & (test_results["preds"] == 0)).sum()),
     int(((test_results["targets"] == 1) & (test_results["preds"] == 1)).sum())],
])
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center", color="black")
ax.set_xticks([0, 1], labels=["Unwatermarked", "Watermarked"])
ax.set_yticks([0, 1], labels=["Unwatermarked", "Watermarked"])
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_title("Held-out test confusion matrix")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# SAVE_CHECKPOINT = True   # Change to True if you want to write the checkpoint to disk
# CHECKPOINT_PATH = Path("resnet50_watermark_baseline.pt")
# if SAVE_CHECKPOINT:
#     torch.save({
#         "model_state_dict": model.state_dict(),
#         "watermark_pattern": watermark_pattern.cpu(),
#         "watermark_strength": WATERMARK_STRENGTH,
#         "image_size": IMAGE_SIZE,
#         "imagenet_mean": IMAGENET_MEAN,
#         "imagenet_std": IMAGENET_STD,
#         "seed": SEED,
#         "history": history.to_dict(orient="records"),
#         "test_metrics": detection_metrics(test_results, "Test"),
#     }, CHECKPOINT_PATH)
#     print("Saved:", CHECKPOINT_PATH.resolve())
# else:
#     print("Checkpoint saving disabled. Set SAVE_CHECKPOINT=True to enable.")

### Implement FGSM/PGD Attack

LLM disclosure:
Code below is written with assistance from ChatGPT with human review to ensure code is correct.

In [ ]:
# FDSM attact

import torch

def fgsm_attack(model, images, labels, epsilon):
    model.eval()

    # Make a copy of the original images
    images = images.clone().detach()

    # Enable gradient calculation for input images
    images.requires_grad = True

    # Forward pass
    outputs = model(images)

    # Calculate classification loss
    loss = torch.nn.functional.cross_entropy(outputs, labels)

    # Clear existing model gradients
    model.zero_grad()

    # Calculate gradients with respect to input images
    loss.backward()

    # Get the sign of the input gradients
    gradient_sign = images.grad.sign()

    # Define normalization values
    mean = torch.tensor(IMAGENET_MEAN, device=images.device).view(1, 3, 1, 1)
    std = torch.tensor(IMAGENET_STD, device=images.device).view(1, 3, 1, 1)

    # Convert epsilon from pixel space to normalized space
    epsilon_normalized = epsilon / std

    # Generate adversarial images
    adv_images = images + epsilon_normalized * gradient_sign

    # Keep pixel values between 0 and 1
    lower_bound = (0 - mean) / std
    upper_bound = (1 - mean) / std

    # Clamp adversarial images to valid pixel boundaries
    adv_images = torch.maximum(
        torch.minimum(adv_images, upper_bound),
        lower_bound
    )

    return adv_images.detach()


In [ ]:
# Find one unwatermarked image and one watermarked image
idx_0 = np.where(trainset.labels == 0)[0][0]
idx_1 = np.where(trainset.labels == 1)[0][0]

# Retrieve the images and their binary labels
image_0, label_0 = trainset[idx_0]
image_1, label_1 = trainset[idx_1]

# Combine them into a batch
images = torch.stack([image_0, image_1]).to(device)
labels = torch.tensor([label_0, label_1]).to(device)

# Apply FGSM attack
epsilon = 0.05
fgsm_adv_images = fgsm_attack(model, images, labels, epsilon)

# Reverse normalization
mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)

fig, axes = plt.subplots(2, 3, figsize=(8, 5))

for i in range(2):
    original = (images[i].detach().cpu() * std + mean).clamp(0, 1)
    adversarial = (fgsm_adv_images[i].detach().cpu() * std + mean).clamp(0, 1)

    difference = (adversarial - original).abs() * 5

    # Original
    axes[i, 0].imshow(original.permute(1, 2, 0))
    axes[i, 0].set_title("Original")

    # FGSM
    axes[i, 1].imshow(adversarial.permute(1, 2, 0))
    axes[i, 1].set_title("FGSM")

    # Difference
    axes[i, 2].imshow(difference.permute(1, 2, 0))
    axes[i, 2].set_title("Difference (amplified)")

    for j in range(3):
        axes[i, j].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# PGD attack
def pgd_attack(model, images, labels, epsilon, alpha, iterations):

    model.eval()
    
    # Keep a copy of the original images
    original_images = images.clone().detach()

    # ImageNet normalization parameters
    mean = torch.tensor(IMAGENET_MEAN, device=images.device).view(1, 3, 1, 1)
    std = torch.tensor(IMAGENET_STD, device=images.device).view(1, 3, 1, 1)

    # Convert epsilon and alpha from pixel space to normalized space
    epsilon_normalized = epsilon / std
    alpha_normalized = alpha / std

    # Valid normalized pixel range
    lower_bound = (0 - mean) / std
    upper_bound = (1 - mean) / std

    # Initialize adversarial images
    adv_images = original_images.clone().detach()

    for i in range(iterations):

        # Enable gradient calculation
        adv_images.requires_grad = True

        # Forward pass
        outputs = model(adv_images)

        # Calculate classification loss
        loss = torch.nn.functional.cross_entropy(outputs, labels)

        # Clear existing model gradients
        model.zero_grad()

        # Calculate gradients with respect to input images
        loss.backward()

        # Get the sign of the gradients
        gradient_sign = adv_images.grad.sign()

        # Update adversarial images
        adv_images = adv_images.detach() + alpha_normalized * gradient_sign

        # Calculate total perturbation
        perturbation = adv_images - original_images

        # Project perturbation into allowed epsilon range
        perturbation = torch.maximum(
            torch.minimum(perturbation, epsilon_normalized),
            -epsilon_normalized
        )

        # Reconstruct adversarial images
        adv_images = original_images + perturbation

        # Keep pixel values within the valid normalized image range
        adv_images = torch.maximum(
            torch.minimum(adv_images, upper_bound),
            lower_bound
        ).detach()

    return adv_images

In [ ]:
# Select one unwatermarked and one watermarked image
idx_0 = np.where(trainset.labels == 0)[0][0]
idx_1 = np.where(trainset.labels == 1)[0][0]

image_0, label_0 = trainset[idx_0]
image_1, label_1 = trainset[idx_1]

# Combine into a batch
images = torch.stack([image_0, image_1]).to(device)
labels = torch.tensor([label_0, label_1]).to(device)

# Apply PGD attack
epsilon = 0.3
alpha = 0.03
num_iter = 10

pgd_adv_images = pgd_attack(
    model, images, labels,
    epsilon, alpha, num_iter
)

# Visualization
fig, axes = plt.subplots(2, 3, figsize=(8, 5))

mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)

for i in range(2):

    # Convert normalized images back to [0, 1]
    original = images[i].detach().cpu()
    adv = pgd_adv_images[i].detach().cpu()

    original = (original * std + mean).clamp(0, 1)
    adv = (adv * std + mean).clamp(0, 1)

    # Calculate amplified difference
    difference = (adv - original).abs() * 5
    difference = difference.clamp(0, 1)

    # Original image
    axes[i, 0].imshow(original.permute(1, 2, 0))
    axes[i, 0].set_title("Original")

    # PGD adversarial image
    axes[i, 1].imshow(adv.permute(1, 2, 0))
    axes[i, 1].set_title("PGD")

    # Difference image
    axes[i, 2].imshow(difference.permute(1, 2, 0))
    axes[i, 2].set_title("Difference (amplified)")

    for j in range(3):
        axes[i, j].axis("off")

    # Label each row
    axes[i, 0].set_ylabel(
        "Unwatermarked" if labels[i].item() == 0 else "Watermarked",
        fontsize=11
    )

plt.tight_layout()
plt.show()

# Alpha: The step size. It determines how much the image changes in each iteration.
# Epsilon: The perturbation budget. It determines the maximum amount the adversarial image can differ from the original image.

### Testing hypotheses

LLM disclosure: Code below is written with assistance from ChatGPT with human review to ensure code is correct.

#### Hypothesis 1: watermark detection accuracy decrease as adversarial perturbation strength increases.
- 1: use baseline resnet50 to obtain accuracy, TPR, and FPR on FGSM / PGD modified test image with increasing epsilon / alpha.

NOTE:

to ensure all hypothesis are tested under the same conditions, set epsilon, alpha, number of iteration, and tolerance to:

epsilon = [0.005, 0.01, 0.02, 0.03]

alpha = [0.001, 0.002, 0.004, 0.006]

num_iter = 10

tolerance = 1e-6

In [ ]:
# baseline resnet50 with FGSM attack
epsilon = [0, 0.005, 0.01, 0.02, 0.03]
tolerance = 1e-6

h1_baseline_fgsm_attack_results = []

for eps in epsilon:

    adv_images_list = []
    labels_list = []

    # Generate FGSM adversarial images
    for images, labels in testloader:

        images = images.to(device)
        labels = labels.to(device)

        adv_images = fgsm_attack(
            model=model,
            images=images,
            labels=labels,
            epsilon=eps
        )

        adv_images_list.append(adv_images.cpu())
        labels_list.append(labels.cpu())

    # Combine all adversarial images
    adv_images = torch.cat(adv_images_list)
    adv_labels = torch.cat(labels_list)

    # Create adversarial test dataset
    adv_test_dataset = torch.utils.data.TensorDataset(
        adv_images, adv_labels
    )

    adv_testloader = torch.utils.data.DataLoader(
        adv_test_dataset,
        batch_size=testloader.batch_size,
        shuffle=False
    )

    # Evaluate baseline model on adversarial images
    fgsm_results = evaluate_model(
        model,
        adv_testloader,
        criterion,
        device
    )

    # Calculate detection metrics
    fgsm_metrics = detection_metrics(
        fgsm_results,
        split_name=f"FGSM epsilon={eps}"
    )

    # Save results
    h1_baseline_fgsm_attack_results.append({
        "epsilon": eps,
        "accuracy": fgsm_results["accuracy"],
        "TPR": fgsm_metrics["TPR / recall"],
        "FPR": fgsm_metrics["FPR"]
    })

    print(f"Finished FGSM evaluation for epsilon = {eps}")

In [ ]:
pd.DataFrame(h1_baseline_fgsm_attack_results)

In [ ]:
# baseline resnet50 with FGSM attack
epsilon = [0, 0.005, 0.01, 0.02, 0.03]
alpha = [0, 0.001, 0.002, 0.004, 0.006]
num_iter = 10
tolerance = 1e-6

h1_baseline_pgd_attack_results = []

for i in range(len(epsilon)):
    eps = epsilon[i]
    alp = alpha[i]

    adv_images_list = []
    labels_list = []

    # Generate PGD adversarial images
    for images, labels in testloader:

        images = images.to(device)
        labels = labels.to(device)

        adv_images = pgd_attack(
            model=model,
            images=images,
            labels=labels,
            alpha=alp,
            iterations=num_iter,
            epsilon=eps
        )

        adv_images_list.append(adv_images.cpu())
        labels_list.append(labels.cpu())

    # Combine all adversarial images
    adv_images = torch.cat(adv_images_list)
    adv_labels = torch.cat(labels_list)

    # Create adversarial test dataset
    adv_test_dataset = torch.utils.data.TensorDataset(
        adv_images, adv_labels
    )

    adv_testloader = torch.utils.data.DataLoader(
        adv_test_dataset,
        batch_size=testloader.batch_size,
        shuffle=False
    )

    # Evaluate baseline model on adversarial images
    pgd_results = evaluate_model(
        model,
        adv_testloader,
        criterion,
        device
    )

    # Calculate detection metrics
    pgd_metrics = detection_metrics(
        pgd_results,
        split_name=f"PGD epsilon={eps}"
    )

    # Save results
    h1_baseline_pgd_attack_results.append({
        "epsilon": eps,
        "accuracy": pgd_results["accuracy"],
        "TPR": pgd_metrics["TPR / recall"],
        "FPR": pgd_metrics["FPR"]
    })

    print(f"Finished PGD evaluation for epsilon = {eps}")

In [ ]:
pd.DataFrame(h1_baseline_pgd_attack_results)

In [ ]:
# combine the two attack results
h1_fgsm_df = pd.DataFrame(h1_baseline_fgsm_attack_results)
h1_pgd_df = pd.DataFrame(h1_baseline_pgd_attack_results)

h1_fgsm_df["attack"] = "FGSM"
h1_pgd_df["attack"] = "PGD"

h1_comparison = pd.concat([h1_fgsm_df, h1_pgd_df], ignore_index=True)

display(h1_comparison)

In [ ]:
list(pd.DataFrame(h1_baseline_fgsm_attack_results)["accuracy"])

In [ ]:
import matplotlib.pyplot as plt

epsilon_values = [0, 0.005, 0.01, 0.02, 0.03]
x_positions = range(len(epsilon_values))


# Plot accuracy
plt.figure(figsize=(4, 3))
fgsm_data = h1_comparison[h1_comparison["attack"] == "FGSM"]
pgd_data = h1_comparison[h1_comparison["attack"] == "PGD"]

plt.plot(x_positions, fgsm_data["accuracy"], marker="o", label="FGSM")
plt.plot(x_positions, pgd_data["accuracy"], marker="s", label="PGD")

plt.xlabel("Perturbation Strength (Epsilon)")
plt.ylabel("Watermark Detection Accuracy")
plt.title("Baseline ResNet50: Clean vs. FGSM vs. PGD")
plt.xticks(x_positions, epsilon_values)
plt.ylim(0, 1)
plt.legend()
plt.grid(True)
plt.show()



# Plot TPR and FPR
metrics = ["TPR", "FPR"]
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
for ax, metric in zip(axes, metrics):
    for attack in ["FGSM", "PGD"]:
        data = h1_comparison[h1_comparison["attack"] == attack]
        ax.plot(x_positions, data[metric], marker="o", label=attack)

    ax.set_xlabel("Perturbation Strength (Epsilon)")
    ax.set_ylabel(metric)
    ax.set_title(f"{metric} vs. Epsilon")
    ax.set_xticks(x_positions)
    ax.set_xticklabels(epsilon_values)
    ax.set_ylim(0, 1)
    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.show()

#### Hypothesis 2: Since PGD uses multiple gradient steps, we hypothesize that it will generally be more effective than FGSM at evading watermark detection
- 1: use baseline watermark detecting resnet50 to predict labels with FGSM / PGD attacked images. (same as hypothsis 1)
    - Should see PGD has lower accuracy
- 2: use FGSM trained resnet50 to predict labels with FGSM / PGD attacked images.
    - Should see PGD has lower accuracy.
- 3: use PGD trained resnet50 to predict labels with FGSM / PGD attacked images.
    - Should see PGD has lower accuracy.

NOTE:

to ensure all hypothesis are tested under the same conditions, set epsilon, alpha, number of iteration, and tolerance to:

epsilon = [0.005, 0.01, 0.02, 0.03]

alpha = [0.001, 0.002, 0.004, 0.006]

num_iter = 10

tolerance = 1e-6

In [ ]:
# baseline watermark detecting resnet50 to predict labels with FGSM / PGD attackted images
# Reuse Hypothesis 1

In [ ]:
# adversarial training FGSM
def train_fgsm_model_cifar10(model, trainloader, valloader, optimizer, criterion,
                              max_epochs, device, epsilon, tolerance=1e-6):
    history = []
    prev_loss = float("inf")
    model.to(device)
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    for epoch in range(max_epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        last_batch_loss = None

        for images, targets in tqdm(trainloader, desc=f"Epoch {epoch+1}/{max_epochs}", leave=False):
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

        # Generate adversarial images
            model.eval()
            adv_images = fgsm_attack(model, images, targets, epsilon)
    
        # Combine original and adversarial images
            combined_images = torch.cat([images, adv_images], dim=0)
            combined_targets = torch.cat([targets, targets], dim=0)

            model.train()
            optimizer.zero_grad(set_to_none=True)

            with torch.autocast("cuda", enabled=use_amp):
                logits = model(combined_images)
                loss = criterion(logits, combined_targets)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            last_batch_loss = loss.item()
            running_loss += last_batch_loss * combined_targets.size(0)
            correct += (logits.argmax(dim=1) == combined_targets).sum().item()
            total += combined_targets.size(0)

        if last_batch_loss is None:
            raise ValueError("Training dataloader is empty.")

        val_results = evaluate_model(model, valloader, criterion, device)
        row = {
            "epoch": epoch + 1,
            "train_loss": running_loss / total,
            "last_batch_loss": last_batch_loss,
            "train_accuracy": correct / total,
            "val_loss": val_results["loss"],
            "val_accuracy": val_results["accuracy"],
        }
        history.append(row)
        print(f"Epoch {epoch+1:02d} | last batch loss {last_batch_loss:.6f} | "
              f"train loss {row['train_loss']:.4f} | "
              f"train acc {row['train_accuracy']:.2%} | "
              f"val loss {row['val_loss']:.4f} | "
              f"val acc {row['val_accuracy']:.2%}")

        if epoch > 0:
            loss_diff = prev_loss - last_batch_loss
            if loss_diff < tolerance:
                print(f"Early stopping at epoch {epoch}")
                break
        prev_loss = last_batch_loss

    return model, pd.DataFrame(history)

In [ ]:
# FGSM trained resnet50 to predict labels with FGSM / PGD attackted images
epsilon = [0.005, 0.01, 0.02, 0.03]
h2_fgsm_models = {}
h2_fgsm_histories = {}

for eps in epsilon:
    # new ResNet50 for FGSM adversarial training
    fgsm_adv_model = models.resnet50(weights=WEIGHTS)

    # Replace classification head
    fgsm_adv_model.fc = nn.Linear(fgsm_adv_model.fc.in_features, 2)

    fgsm_adv_model = fgsm_adv_model.to(device)

    # Create a new optimizer
    fgsm_adv_optimizer = torch.optim.SGD(
        fgsm_adv_model.parameters(),
        lr=LEARNING_RATE,
        momentum=MOMENTUM
    )

    fgsm_model, fgsm_history = train_fgsm_model_cifar10(
        model=fgsm_adv_model,
        trainloader=trainloader,
        valloader=valloader,
        optimizer=fgsm_adv_optimizer,
        criterion=criterion,
        max_epochs=10,
        device=device,
        epsilon=eps,
        tolerance=1e-6
    )

    # Save model and training history
    h2_fgsm_models[eps] = fgsm_model
    h2_fgsm_histories[eps] = fgsm_history

    print(f"Finished training epsilon = {eps}")

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
h2_fgsm_results = []

attack_epsilon = [0.005, 0.01, 0.02, 0.03]
attack_alpha =  [0.001, 0.002, 0.004, 0.006]

for training_eps, fgsm_model in h2_fgsm_models.items():

    print(f"\nEvaluating FGSM model trained with epsilon = {training_eps}")

    for eps, alp in zip(attack_epsilon, attack_alpha):
        # Generate FGSM adversarial test images
        fgsm_adv_images_list = []
        fgsm_labels_list = []

        for images, labels in testloader:
            images = images.to(device)
            labels = labels.to(device)

            adv_images = fgsm_attack(
                fgsm_model,
                images,
                labels,
                eps
            )

            fgsm_adv_images_list.append(adv_images.cpu())
            fgsm_labels_list.append(labels.cpu())

        # Combine FGSM images and labels
        fgsm_adv_images = torch.cat(fgsm_adv_images_list)
        fgsm_adv_labels = torch.cat(fgsm_labels_list)

        # Create FGSM adversarial test DataLoader
        fgsm_test_dataset = TensorDataset(
            fgsm_adv_images,
            fgsm_adv_labels
        )

        fgsm_testloader = DataLoader(
            fgsm_test_dataset,
            batch_size=testloader.batch_size,
            shuffle=False
        )

        # Evaluate FGSM adversarial images
        fgsm_test_results = evaluate_model(
            fgsm_model,
            fgsm_testloader,
            criterion,
            device
        )

        fgsm_metrics = detection_metrics( fgsm_test_results, "FGSM Test")

        h2_fgsm_results.append({
            "training_epsilon": training_eps,
            "attack_epsilon": eps,
            "attack": "FGSM",
            "accuracy": fgsm_test_results["accuracy"],
            "TPR": fgsm_metrics["TPR / recall"],
            "FPR": fgsm_metrics["FPR"],
            "precision": fgsm_metrics["precision"]
        })

        # Generate PGD adversarial test images
        pgd_adv_images_list = []
        pgd_labels_list = []

        for images, labels in testloader:
            images = images.to(device)
            labels = labels.to(device)

            adv_images = pgd_attack(
                fgsm_model,
                images,
                labels,
                epsilon=eps,
                alpha=alp,
                iterations=10
            )

            pgd_adv_images_list.append(adv_images.cpu())
            pgd_labels_list.append(labels.cpu())

        # Combine PGD images and labels
        pgd_adv_images = torch.cat(pgd_adv_images_list)
        pgd_adv_labels = torch.cat(pgd_labels_list)

        # Create PGD adversarial test DataLoader
        pgd_test_dataset = TensorDataset(
            pgd_adv_images,
            pgd_adv_labels
        )

        pgd_testloader = DataLoader(
            pgd_test_dataset,
            batch_size=testloader.batch_size,
            shuffle=False
        )

        # Evaluate PGD adversarial images
        pgd_test_results = evaluate_model(
            fgsm_model,
            pgd_testloader,
            criterion,
            device
        )

        pgd_metrics = detection_metrics(pgd_test_results, "PGD Test")

        h2_fgsm_results.append({
            "training_epsilon": training_eps,
            "attack_epsilon": eps,
            "attack": "PGD",
            "accuracy": pgd_test_results["accuracy"],
            "TPR": pgd_metrics["TPR / recall"],
            "FPR": pgd_metrics["FPR"],
            "precision": pgd_metrics["precision"]
        })

# Convert results into a DataFrame
h2_fgsm_results = pd.DataFrame(h2_fgsm_results)

In [ ]:
h2_fgsm_results.head(10)

In [ ]:
plt.figure(figsize=(7,4))

for attack in ["FGSM", "PGD"]:
    subset = h2_fgsm_results[
        h2_fgsm_results["attack"] == attack
    ]

    for training_eps, group in subset.groupby("training_epsilon"):
        group = group.sort_values("attack_epsilon")

        plt.plot(
            group["attack_epsilon"],
            group["accuracy"],
            marker="o",
            linestyle="-" if attack == "FGSM" else "--",
            label=f"{attack}, Training ε = {training_eps}"
        )

plt.xlabel("Attack Epsilon")
plt.ylabel("Accuracy")
plt.title("FGSM Trained Model's Accuracy Under FGSM and PGD Attacks")
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# adversarial training PGD
def train_pgd_model_cifar10(model, trainloader, valloader, optimizer, criterion,
                              max_epochs, device, epsilon, alpha, num_iter, tolerance=1e-6):
    history = []
    prev_loss = float("inf")
    model.to(device)
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    for epoch in range(max_epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        last_batch_loss = None

        for images, targets in tqdm(trainloader, desc=f"Epoch {epoch+1}/{max_epochs}", leave=False):
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            # Generate adversarial images
            model.eval()
            adv_images = pgd_attack(model, images, targets, epsilon, alpha, num_iter)
    
            # Combine original and adversarial images
            combined_images = torch.cat([images, adv_images], dim=0)
            combined_targets = torch.cat([targets, targets], dim=0)

            model.train()
            optimizer.zero_grad(set_to_none=True)

            with torch.autocast("cuda", enabled=use_amp):
                logits = model(combined_images)
                loss = criterion(logits, combined_targets)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            last_batch_loss = loss.item()
            running_loss += last_batch_loss * combined_targets.size(0)
            correct += (logits.argmax(dim=1) == combined_targets).sum().item()
            total += combined_targets.size(0)

        if last_batch_loss is None:
            raise ValueError("Training dataloader is empty.")

        val_results = evaluate_model(model, valloader, criterion, device)
        row = {
            "epoch": epoch + 1,
            "train_loss": running_loss / total,
            "last_batch_loss": last_batch_loss,
            "train_accuracy": correct / total,
            "val_loss": val_results["loss"],
            "val_accuracy": val_results["accuracy"],
        }
        history.append(row)
        print(f"Epoch {epoch+1:02d} | last batch loss {last_batch_loss:.6f} | "
              f"train loss {row['train_loss']:.4f} | "
              f"train acc {row['train_accuracy']:.2%} | "
              f"val loss {row['val_loss']:.4f} | "
              f"val acc {row['val_accuracy']:.2%}")

        if epoch > 0:
            loss_diff = prev_loss - last_batch_loss
            if loss_diff < tolerance:
                print(f"Early stopping at epoch {epoch}")
                break
        prev_loss = last_batch_loss

    return model, pd.DataFrame(history)

In [ ]:
# FGSM trained resnet50 to predict labels with FGSM / PGD attackted images
epsilon = [0.005, 0.01, 0.02, 0.03]
alpha =  [0.001, 0.002, 0.004, 0.006]
num_iter = 10

h2_pgd_models = {}
h2_pgd_histories = {}

for eps, alp in zip(epsilon, alpha):
    print(f"\nTraining PGD model with epsilon = {eps}, alpha = {alp}")
    
    # new ResNet50 for PGD adversarial training
    pgd_adv_model = models.resnet50(weights=WEIGHTS)

    # Replace classification head
    pgd_adv_model.fc = nn.Linear(pgd_adv_model.fc.in_features, 2)

    pgd_adv_model = pgd_adv_model.to(device)

    # Create a new optimizer
    pgd_adv_optimizer = torch.optim.SGD(
        pgd_adv_model.parameters(),
        lr=LEARNING_RATE,
        momentum=MOMENTUM
    )

    pgd_model, pgd_history = train_pgd_model_cifar10(
        model=pgd_adv_model,
        trainloader=trainloader,
        valloader=valloader,
        optimizer=pgd_adv_optimizer,
        criterion=criterion,
        max_epochs=10,
        device=device,
        epsilon=eps,
        alpha=alp,
        num_iter=num_iter,
        tolerance=1e-6
    )

    # Save model and training history
    h2_pgd_models[eps] = pgd_model
    h2_pgd_histories[eps] = pgd_history

    print(f"Finished training epsilon = {eps}")

In [ ]:

h2_pgd_results = []

attack_epsilon = [0.005, 0.01, 0.02, 0.03]
attack_alpha = [0.001, 0.006]
num_iter = 10

training_alpha = {
    0.005: 0.001,
    0.01: 0.002,
    0.02: 0.004,
    0.03: 0.006
}

for training_eps, pgd_model in h2_pgd_models.items():

    print(f"\nEvaluating PGD model trained with epsilon = {training_eps}")

    for eps in attack_epsilon:

        # Generate FGSM adversarial test images
        fgsm_adv_images_list = []
        fgsm_labels_list = []

        for images, labels in testloader:
            images = images.to(device)
            labels = labels.to(device)

            adv_images = fgsm_attack(
                pgd_model,
                images,
                labels,
                eps
            )

            fgsm_adv_images_list.append(adv_images.cpu())
            fgsm_labels_list.append(labels.cpu())

        # Combine FGSM images and labels
        fgsm_adv_images = torch.cat(fgsm_adv_images_list)
        fgsm_adv_labels = torch.cat(fgsm_labels_list)

        # Create FGSM adversarial test DataLoader
        fgsm_test_dataset = TensorDataset(
            fgsm_adv_images,
            fgsm_adv_labels
        )

        fgsm_testloader = DataLoader(
            fgsm_test_dataset,
            batch_size=testloader.batch_size,
            shuffle=False
        )

        # Evaluate FGSM adversarial images
        fgsm_test_results = evaluate_model(
            pgd_model,
            fgsm_testloader,
            criterion,
            device
        )

        fgsm_metrics = detection_metrics(
            fgsm_test_results, "FGSM Test"
        )

        h2_pgd_results.append({
            "training_epsilon": training_eps,
            "training_alpha": training_alpha[training_eps],
            "attack_epsilon": eps,
            "attack_alpha": None,
            "attack": "FGSM",
            "accuracy": fgsm_test_results["accuracy"],
            "TPR": fgsm_metrics["TPR / recall"],
            "FPR": fgsm_metrics["FPR"],
            "precision": fgsm_metrics["precision"]
        })

        # Generate PGD adversarial test images
        for alp in attack_alpha:

            pgd_adv_images_list = []
            pgd_labels_list = []

            for images, labels in testloader:
                images = images.to(device)
                labels = labels.to(device)

                adv_images = pgd_attack(
                    pgd_model,
                    images,
                    labels,
                    epsilon=eps,
                    alpha=alp,
                    iterations=num_iter
                )

                pgd_adv_images_list.append(adv_images.cpu())
                pgd_labels_list.append(labels.cpu())

            # Combine PGD images and labels
            pgd_adv_images = torch.cat(pgd_adv_images_list)
            pgd_adv_labels = torch.cat(pgd_labels_list)

            # Create PGD adversarial test DataLoader
            pgd_test_dataset = TensorDataset(
                pgd_adv_images,
                pgd_adv_labels
            )

            pgd_testloader = DataLoader(
                pgd_test_dataset,
                batch_size=testloader.batch_size,
                shuffle=False
            )

            # Evaluate PGD adversarial images
            pgd_test_results = evaluate_model(
                pgd_model,
                pgd_testloader,
                criterion,
                device
            )

            pgd_metrics = detection_metrics(
                pgd_test_results, "PGD Test"
            )

            h2_pgd_results.append({
                "training_epsilon": training_eps,
                "training_alpha": training_alpha[training_eps],
                "attack_epsilon": eps,
                "attack_alpha": alp,
                "attack": "PGD",
                "accuracy": pgd_test_results["accuracy"],
                "TPR": pgd_metrics["TPR / recall"],
                "FPR": pgd_metrics["FPR"],
                "precision": pgd_metrics["precision"]
            })

# Convert results into a DataFrame
h2_pgd_results = pd.DataFrame(h2_pgd_results)


In [ ]:
print(h2_pgd_results.shape)
display(h2_pgd_results)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

training_epsilons = [0.005, 0.01, 0.02, 0.03]
attack_epsilons = [0.005, 0.01, 0.02, 0.03]

results = h2_pgd_results

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
axes = axes.flatten()

for ax, training_eps in zip(axes, training_epsilons):

    subset = results[
        results["training_epsilon"] == training_eps
    ]

    # FGSM accuracy
    fgsm_values = []
    for eps in attack_epsilons:
        accuracy = subset[
            (subset["attack"] == "FGSM") &
            (subset["attack_epsilon"] == eps)
        ]["accuracy"].iloc[0]
        fgsm_values.append(accuracy)

    # PGD accuracy, alpha = 0.001
    pgd_001_values = []
    for eps in attack_epsilons:
        accuracy = subset[
            (subset["attack"] == "PGD") &
            (subset["attack_epsilon"] == eps) &
            (subset["attack_alpha"] == 0.001)
        ]["accuracy"].iloc[0]
        pgd_001_values.append(accuracy)

    # PGD accuracy, alpha = 0.006
    pgd_006_values = []
    for eps in attack_epsilons:
        accuracy = subset[
            (subset["attack"] == "PGD") &
            (subset["attack_epsilon"] == eps) &
            (subset["attack_alpha"] == 0.006)
        ]["accuracy"].iloc[0]
        pgd_006_values.append(accuracy)

    # Plot the three lines
    ax.plot(
        attack_epsilons, fgsm_values,
        marker="o", linewidth=2, label="FGSM"
    )

    ax.plot(
        attack_epsilons, pgd_001_values,
        marker="s", linewidth=2, label="PGD (alpha = 0.001)"
    )

    ax.plot(
        attack_epsilons, pgd_006_values,
        marker="^", linewidth=2, label="PGD (alpha = 0.006)"
    )

    ax.set_title(f"Training Epsilon = {training_eps} & Training Alpha = {training_eps*2/10}")
    ax.set_xlabel("Attack Epsilon & Attack Alpha if applicable")
    ax.set_ylabel("Accuracy")
    ax.set_xticks(attack_epsilons)
    ax.set_ylim(0, 0.5)
    ax.grid(alpha=0.3)
    ax.legend()

fig.suptitle(
    "PGD-Trained Model Accuracy Under FGSM and PGD Attacks",
    fontsize=14
)

plt.tight_layout()
plt.show()

#### Hypothesis 3: adversarially trained detectors to be more resistant to these attacks than conventionally trained detectors.
- 1: use results from hypothesis 1 and hypohethesis 2 to get the difference between baseline resnet50's performance on FGSM and PGD attack and the FGSM and PGD trained resnet50's performance on increasing epsilon / alpha for attacking FGSM and PGD.

NOTE:
to ensure all hypothesis are tested under the same conditions, set epsilon, alpha, number of iteration, and tolerance to:

epsilon = [0.005, 0.01, 0.02, 0.03]

alpha = [0.001, 0.002, 0.004, 0.006]

num_iter = 10

tolerance = 1e-6


In [ ]:
# reuse things from hypothesis 1 and 2, but need to rerun the whole notebook with full training and testing data.
# then use the dataframes from each part to combine into a plot to show our findings

#### Hypothesis 4: stronger watermark patterns to improve detection robustness at a tradeoff of image fidelity
- 1: use very small to large (0.005, 0.01, 0.02, 0.05) watermarked images to train resnet50, and test with watermarking only, FGSM attack, and PGD attack images.

NOTE:

to ensure all hypothesis are tested under the same conditions, set epsilon, alpha, number of iteration, and tolerance to:

epsilon = [0.005, 0.01, 0.02, 0.03]

alpha = [0.001, 0.002, 0.004, 0.006]

num_iter = 10

tolerance = 1e-6

In [ ]:
watermark_strengths = [0.005, 0.01, 0.02, 0.05]

attack_epsilon = [0.005, 0.01, 0.02, 0.03]
attack_alpha = [0.001, 0.006]
num_iter = 10

In [ ]:
# reuse previous parts but modify it so can change watermark strength to create different watermark strength datasets
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

# Watermark strengths for H4
watermark_strengths = [0.005, 0.01, 0.02, 0.05]


# Inputs are PIL originals; watermark is injected before resizing and normalization.
class WatermarkCIFAR10(Dataset):
    def __init__(self, cifar_dataset, indices, strength=WATERMARK_STRENGTH,
                 pattern=watermark_pattern, seed=42, image_size=32):

        self.base = cifar_dataset
        self.indices = np.asarray(indices, dtype=int)
        self.strength = strength
        self.pattern = pattern
        self.image_size = image_size

        self.labels = np.zeros(len(self.indices), dtype=np.int64)
        object_labels = np.asarray(self.base.targets)[self.indices]

        rng = np.random.default_rng(seed)

        for cls in np.unique(object_labels):
            positions = np.flatnonzero(object_labels == cls)

            if len(positions) % 2:
                raise ValueError("Each source class must have even count for a 50/50 label split")

            rng.shuffle(positions)
            self.labels[positions[:len(positions) // 2]] = 1

        self.resize = transforms.Resize((image_size, image_size), antialias=True)
        self.normalize = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = int(self.indices[i])

        # Load directly from CIFAR-10's NumPy data
        image = self.base.data[idx]

        # HWC uint8 -> CHW float32 in [0, 1]
        x = (
            torch.from_numpy(image)
            .permute(2, 0, 1)
            .float()
            .div_(255.0)
        )

        y = int(self.labels[i])

        if y == 1:
            x = add_watermark(
                x,
                strength=self.strength,
                pattern=self.pattern
            )

        # Only resize when necessary
        if self.image_size != 32:
            x = self.resize(x)

        x = self.normalize(x)

        return x, torch.tensor(y, dtype=torch.long)


# Create datasets for each watermark strength
h4_datasets = {}

for strength in watermark_strengths:

    trainset_h4 = WatermarkCIFAR10(
        cifar_train,
        train_indices,
        strength=strength,
        pattern=watermark_pattern,
        seed=SEED + 10
    )

    valset_h4 = WatermarkCIFAR10(
        cifar_train,
        val_indices,
        strength=strength,
        pattern=watermark_pattern,
        seed=SEED + 11
    )

    testset_h4 = WatermarkCIFAR10(
        cifar_test,
        test_indices,
        strength=strength,
        pattern=watermark_pattern,
        seed=SEED + 12
    )

    trainloader_h4 = DataLoader(
        trainset_h4,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=device.type == "cuda"
    )

    valloader_h4 = DataLoader(
        valset_h4,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=device.type == "cuda"
    )

    testloader_h4 = DataLoader(
        testset_h4,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=device.type == "cuda"
    )

    h4_datasets[strength] = {
        "train": trainset_h4,
        "val": valset_h4,
        "test": testset_h4,
        "trainloader": trainloader_h4,
        "valloader": valloader_h4,
        "testloader": testloader_h4
    }


# Display dataset summary
summary = []

for strength, datasets in h4_datasets.items():

    for name in ["train", "val", "test"]:

        ds = datasets[name]

        summary.append({
            "watermark_strength": strength,
            "partition": name,
            "unwatermarked": int(np.sum(ds.labels == 0)),
            "watermarked": int(np.sum(ds.labels == 1)),
            "total": len(ds)
        })

h4_summary = pd.DataFrame(summary)

display(h4_summary)

# Verify balanced labels
assert all(
    row.unwatermarked == row.watermarked
    for row in h4_summary.itertuples()
)

print("H4 datasets successfully created!")


In [ ]:
h4_models = {}
h4_histories = {}

for strength in watermark_strengths:

    print(f"\nTraining model with watermark strength = {strength}")
    trainloader_h4 = h4_datasets[strength]["trainloader"]
    valloader_h4 = h4_datasets[strength]["valloader"]

    model = models.resnet50(weights=WEIGHTS)
    model.fc = nn.Linear(model.fc.in_features, 2)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    LEARNING_RATE = 0.001
    MOMENTUM = 0.9
    MAX_EPOCHS = 15
    TOLERANCE = 1e-6

    optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=MOMENTUM)

    trained_model, history = train_torch_model_cifar10(
    model=model,
    trainloader=trainloader_h4,
    valloader=valloader_h4,
    optimizer=optimizer,
    criterion=criterion,
    max_epochs=MAX_EPOCHS,
    device=device,
    tolerance=TOLERANCE,
    )

    h4_models[strength] = trained_model
    h4_histories[strength] = history

In [ ]:
# evaluate the watermark only performance on models
h4_results = []

for strength, model in h4_models.items():

    print(f"\nEvaluating watermark strength = {strength}")

    testloader_h4 = h4_datasets[strength]["testloader"]

    clean_results = evaluate_model(
        model,
        testloader_h4,
        criterion,
        device
    )

    clean_metrics = detection_metrics(
        clean_results,
        "Clean Watermarked Test"
    )

    h4_results.append({
        "watermark_strength": strength,
        "attack": "Clean",
        "attack_epsilon": None,
        "attack_alpha": None,
        "accuracy": clean_results["accuracy"],
        "TPR": clean_metrics["TPR / recall"],
        "FPR": clean_metrics["FPR"],
        "precision": clean_metrics["precision"]
    })

In [ ]:
# evaluate FGSM attacked images on model performance
for strength, model in h4_models.items():

    print(f"\nEvaluating FGSM: watermark strength = {strength}")

    testloader_h4 = h4_datasets[strength]["testloader"]

    for eps in attack_epsilon:

        adv_images_list = []
        adv_labels_list = []

        for images, labels in testloader_h4:

            images = images.to(device)
            labels = labels.to(device)

            adv_images = fgsm_attack(
                model,
                images,
                labels,
                epsilon=eps
            )

            adv_images_list.append(adv_images.cpu())
            adv_labels_list.append(labels.cpu())

        adv_images = torch.cat(adv_images_list)
        adv_labels = torch.cat(adv_labels_list)

        adv_dataset = TensorDataset(adv_images, adv_labels)

        adv_testloader = DataLoader(
            adv_dataset,
            batch_size=testloader_h4.batch_size,
            shuffle=False
        )

        adv_results = evaluate_model(
            model,
            adv_testloader,
            criterion,
            device
        )

        adv_metrics = detection_metrics(
            adv_results,
            "FGSM Test"
        )

        h4_results.append({
            "watermark_strength": strength,
            "attack": "FGSM",
            "attack_epsilon": eps,
            "attack_alpha": None,
            "accuracy": adv_results["accuracy"],
            "TPR": adv_metrics["TPR / recall"],
            "FPR": adv_metrics["FPR"],
            "precision": adv_metrics["precision"]
        })

In [ ]:
# evaluate PGD attacked images on model performance
for strength, model in h4_models.items():

    print(f"\nEvaluating PGD: watermark strength = {strength}")

    testloader_h4 = h4_datasets[strength]["testloader"]

    for eps in attack_epsilon:

        for alp in attack_alpha:

            adv_images_list = []
            adv_labels_list = []

            for images, labels in testloader_h4:

                images = images.to(device)
                labels = labels.to(device)

                adv_images = pgd_attack(
                    model,
                    images,
                    labels,
                    epsilon=eps,
                    alpha=alp,
                    iterations=num_iter
                )

                adv_images_list.append(adv_images.cpu())
                adv_labels_list.append(labels.cpu())

            adv_images = torch.cat(adv_images_list)
            adv_labels = torch.cat(adv_labels_list)

            adv_dataset = TensorDataset(
                adv_images,
                adv_labels
            )

            adv_testloader = DataLoader(
                adv_dataset,
                batch_size=testloader_h4.batch_size,
                shuffle=False
            )

            adv_results = evaluate_model(
                model,
                adv_testloader,
                criterion,
                device
            )

            adv_metrics = detection_metrics(
                adv_results,
                "PGD Test"
            )

            h4_results.append({
                "watermark_strength": strength,
                "attack": "PGD",
                "attack_epsilon": eps,
                "attack_alpha": alp,
                "accuracy": adv_results["accuracy"],
                "TPR": adv_metrics["TPR / recall"],
                "FPR": adv_metrics["FPR"],
                "precision": adv_metrics["precision"]
            })

In [ ]:
h4_results = pd.DataFrame(h4_results)

display(
    h4_results.sort_values(
        by=[
            "watermark_strength",
            "attack",
            "attack_epsilon",
            "attack_alpha"
        ]
    ).reset_index(drop=True)
)

In [ ]:
# plot results
import matplotlib.pyplot as plt
import numpy as np

watermark_strengths = [0.005, 0.01, 0.02, 0.05]
attack_epsilons = [0.005, 0.01, 0.02, 0.03]

results = h4_results

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, strength in zip(axes, watermark_strengths):

    subset = results[results["watermark_strength"] == strength]

    # Clean accuracy (one value, repeated for visual comparison)
    clean_accuracy = subset[subset["attack"] == "Clean"]["accuracy"].iloc[0]
    clean_values = [clean_accuracy] * len(attack_epsilons)

    # FGSM accuracy
    fgsm_values = []
    for eps in attack_epsilons:
        accuracy = subset[
            (subset["attack"] == "FGSM") &
            (subset["attack_epsilon"] == eps)
        ]["accuracy"].iloc[0]

        fgsm_values.append(accuracy)

    # PGD alpha = 0.001
    pgd_001_values = []
    for eps in attack_epsilons:
        accuracy = subset[
            (subset["attack"] == "PGD") &
            (subset["attack_epsilon"] == eps) &
            (subset["attack_alpha"] == 0.001)
        ]["accuracy"].iloc[0]

        pgd_001_values.append(accuracy)

    # PGD alpha = 0.006
    pgd_006_values = []
    for eps in attack_epsilons:
        accuracy = subset[
            (subset["attack"] == "PGD") &
            (subset["attack_epsilon"] == eps) &
            (subset["attack_alpha"] == 0.006)
        ]["accuracy"].iloc[0]

        pgd_006_values.append(accuracy)

    ax.plot(attack_epsilons, clean_values,
            marker="o", linewidth=2, label="Clean")

    ax.plot(attack_epsilons, fgsm_values,
            marker="s", linewidth=2, label="FGSM")

    ax.plot(attack_epsilons, pgd_001_values,
            marker="^", linewidth=2, label="PGD (alpha = 0.001)")

    ax.plot(attack_epsilons, pgd_006_values,
            marker="D", linewidth=2, label="PGD (alpha = 0.006)")

    ax.set_title(f"Watermark Strength = {strength}")
    ax.set_xlabel("Attack Epsilon")
    ax.set_ylabel("Detection Accuracy")
    ax.set_xticks(attack_epsilons)
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    ax.legend()

fig.suptitle(
    "Watermark Detection Accuracy Under Different Attack Conditions",
    fontsize=15
)

plt.tight_layout()
plt.show()

##### please ignore the rest of the notebook, these are just separate pieces of code that I was running to test/figure things out, can delete all parts after this cell before turning in our code.

### Adversarial training (FGSM)
LLM disclosure:
Code below is written with assistance from ChatGPT with human review to ensure code is correct.

In [ ]:
# # FGSM attack
# def train_fgsm_model_cifar10(model, trainloader, valloader, optimizer, criterion,
#                               max_epochs, device, epsilon, tolerance=1e-6):
#     history = []
#     prev_loss = float("inf")
#     model.to(device)
#     use_amp = device.type == "cuda"
#     scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

#     for epoch in range(max_epochs):
#         model.train()
#         running_loss, correct, total = 0.0, 0, 0
#         last_batch_loss = None

#         for images, targets in tqdm(trainloader, desc=f"Epoch {epoch+1}/{max_epochs}", leave=False):
#             images = images.to(device, non_blocking=True)
#             targets = targets.to(device, non_blocking=True)

#         # Generate adversarial images
#             model.eval()
#             adv_images = fgsm_attack(model, images, targets, epsilon)
    
#         # Combine original and adversarial images
#             combined_images = torch.cat([images, adv_images], dim=0)
#             combined_targets = torch.cat([targets, targets], dim=0)

#             model.train()
#             optimizer.zero_grad(set_to_none=True)

#             with torch.autocast("cuda", enabled=use_amp):
#                 logits = model(combined_images)
#                 loss = criterion(logits, combined_targets)

#             scaler.scale(loss).backward()
#             scaler.step(optimizer)
#             scaler.update()

#             last_batch_loss = loss.item()
#             running_loss += last_batch_loss * combined_targets.size(0)
#             correct += (logits.argmax(dim=1) == combined_targets).sum().item()
#             total += combined_targets.size(0)

#         if last_batch_loss is None:
#             raise ValueError("Training dataloader is empty.")

#         val_results = evaluate_model(model, valloader, criterion, device)
#         row = {
#             "epoch": epoch + 1,
#             "train_loss": running_loss / total,
#             "last_batch_loss": last_batch_loss,
#             "train_accuracy": correct / total,
#             "val_loss": val_results["loss"],
#             "val_accuracy": val_results["accuracy"],
#         }
#         history.append(row)
#         print(f"Epoch {epoch+1:02d} | last batch loss {last_batch_loss:.6f} | "
#               f"train loss {row['train_loss']:.4f} | "
#               f"train acc {row['train_accuracy']:.2%} | "
#               f"val loss {row['val_loss']:.4f} | "
#               f"val acc {row['val_accuracy']:.2%}")

#         if epoch > 0:
#             loss_diff = prev_loss - last_batch_loss
#             if loss_diff < tolerance:
#                 print(f"Early stopping at epoch {epoch}")
#                 break
#         prev_loss = last_batch_loss

#     return model, pd.DataFrame(history)

In [ ]:
# # new ResNet50 for FGSM adversarial training
# fgsm_adv_model = models.resnet50(weights=WEIGHTS)

# # Replace classification head
# fgsm_adv_model.fc = nn.Linear(fgsm_adv_model.fc.in_features, 2)

# fgsm_adv_model = fgsm_adv_model.to(device)

# # Create a new optimizer
# fgsm_adv_optimizer = torch.optim.SGD(
#     fgsm_adv_model.parameters(),
#     lr=LEARNING_RATE,
#     momentum=MOMENTUM
# )

# print("FGSM Model:", type(fgsm_adv_model).__name__)
# print("Classification head:", fgsm_adv_model.fc)

In [ ]:
# epsilon = [0.005, 0.01, 0.02, 0.03]
# all_fgsm_results = []

# for eps in epsilon:
#     # new ResNet50 for FGSM adversarial training
#     fgsm_adv_model = models.resnet50(weights=WEIGHTS)

#     # Replace classification head
#     fgsm_adv_model.fc = nn.Linear(fgsm_adv_model.fc.in_features, 2)

#     fgsm_adv_model = fgsm_adv_model.to(device)

#     # Create a new optimizer
#     fgsm_adv_optimizer = torch.optim.SGD(
#         fgsm_adv_model.parameters(),
#         lr=LEARNING_RATE,
#         momentum=MOMENTUM
#     )

#     fgsm_model, fgsm_history = train_fgsm_model_cifar10(
#         model=fgsm_adv_model,
#         trainloader=trainloader,
#         valloader=valloader,
#         optimizer=fgsm_adv_optimizer,
#         criterion=criterion,
#         max_epochs=10,
#         device=device,
#         epsilon=eps,
#         tolerance=1e-6
#     )

#     # display epoch history
#     display(fgsm_history)


#     # Evaluate the FGSM-trained model on clean images
#     train_results = evaluate_model(fgsm_model, trainloader, criterion, device)
#     val_results = evaluate_model(fgsm_model, valloader, criterion, device)
#     test_results = evaluate_model(fgsm_model, testloader, criterion, device)

#     # Create a metrics table
#     clean_metrics = pd.DataFrame([
#         detection_metrics(train_results, "Train"),
#         detection_metrics(val_results, "Validation"),
#         detection_metrics(test_results, "Test"),
#     ])

#     print("FGSM-TRAINED MODEL: CLEAN IMAGE PERFORMANCE")

#     display(clean_metrics.style.format({
#         "loss": "{:.4f}",
#         "accuracy": "{:.2%}",
#         "TPR / recall": "{:.2%}",
#         "FPR": "{:.2%}",
#         "precision": "{:.2%}",
#     }))

#     print(f"Test accuracy: {test_results['accuracy']:.2%}")

#     from torch.utils.data import TensorDataset, DataLoader

#     fgsm_adv_images_list = []
#     fgsm_labels_list = []

#     # Generate FGSM adversarial images from the test set
#     for images, labels in testloader:
#         images = images.to(device)
#         labels = labels.to(device)

#         adv_images = fgsm_attack(
#             fgsm_model,
#             images,
#             labels,
#             eps
#         )

#         fgsm_adv_images_list.append(adv_images.cpu())
#         fgsm_labels_list.append(labels.cpu())

#     # Combine all adversarial images and labels
#     fgsm_adv_images = torch.cat(fgsm_adv_images_list)
#     fgsm_adv_labels = torch.cat(fgsm_labels_list)

#     # Create a DataLoader for the adversarial test set
#     fgsm_test_dataset = TensorDataset(fgsm_adv_images, fgsm_adv_labels)
#     fgsm_testloader = DataLoader(
#         fgsm_test_dataset,
#         batch_size=testloader.batch_size,
#         shuffle=False
#     )

#     # Evaluate the FGSM-trained model on adversarial test images
#     fgsm_test_results = evaluate_model(
#         fgsm_model,
#         fgsm_testloader,
#         criterion,
#         device
#     )

#     # Display adversarial test metrics
#     fgsm_metrics = pd.DataFrame([
#         detection_metrics(fgsm_test_results, "FGSM-attacked Test")
#     ])

#     print("FGSM-TRAINED MODEL: ADVERSARIAL PERFORMANCE")

#     display(fgsm_metrics.style.format({
#         "loss": "{:.4f}",
#         "accuracy": "{:.2%}",
#         "TPR / recall": "{:.2%}",
#         "FPR": "{:.2%}",
#         "precision": "{:.2%}",
#     }))

#     print(f"Accuracy under FGSM attack: {fgsm_test_results['accuracy']:.2%}")
#     print(f"TPR under FGSM attack: {fgsm_metrics.iloc[0]['TPR / recall']:.2%}")
#     print(f"FPR under FGSM attack: {fgsm_metrics.iloc[0]['FPR']:.2%}")

#     # How often the attack makes a correctly detected watermark go undetected
#     import numpy as np

#     clean_targets = np.asarray(test_results["targets"])
#     clean_preds = np.asarray(test_results["preds"])
#     adv_preds = np.asarray(fgsm_test_results["preds"])

#     # Select watermarked images correctly detected before the attack
#     correct_watermarked = (clean_targets == 1) & (clean_preds == 1)

#     # Count how many are classified as unwatermarked after the attack
#     successful_attacks = correct_watermarked & (adv_preds == 0)

#     if correct_watermarked.sum() > 0:
#         attack_success_rate = successful_attacks.sum() / correct_watermarked.sum()
#     else:
#         attack_success_rate = float("nan")

#     print(f"FGSM Attack Success Rate: {attack_success_rate:.2%}")


#     # save all results in a list
#     all_fgsm_results.append({
#     "training_epsilon": eps,
#     "clean_accuracy": test_results["accuracy"],
#     "adversarial_accuracy": fgsm_test_results["accuracy"],
#     "clean_TPR": clean_metrics.iloc[2]["TPR / recall"],
#     "adversarial_TPR": fgsm_metrics.iloc[0]["TPR / recall"],
#     "clean_FPR": clean_metrics.iloc[2]["FPR"],
#     "adversarial_FPR": fgsm_metrics.iloc[0]["FPR"],
#     "attack_success_rate": attack_success_rate
#     })



#     print("####################### NEW SET OF EPSILON AND ALPHA #############################################")

        

In [ ]:
# # show results table
# fgsm_comparison = pd.DataFrame(all_fgsm_results)

# display(fgsm_comparison.style.format({
#     "training_epsilon": "{:.3f}",
#     "clean_accuracy": "{:.2%}",
#     "adversarial_accuracy": "{:.2%}",
#     "clean_TPR": "{:.2%}",
#     "adversarial_TPR": "{:.2%}",
#     "clean_FPR": "{:.2%}",
#     "adversarial_FPR": "{:.2%}",
#     "attack_success_rate": "{:.2%}"
# }))

In [ ]:
# display(fgsm_history)

In [ ]:
# # Evaluate the FGSM-trained model on clean images
# train_results = evaluate_model(fgsm_adv_model, trainloader, criterion, device)
# val_results = evaluate_model(fgsm_adv_model, valloader, criterion, device)
# test_results = evaluate_model(fgsm_adv_model, testloader, criterion, device)

# # Create a metrics table
# clean_metrics = pd.DataFrame([
#     detection_metrics(train_results, "Train"),
#     detection_metrics(val_results, "Validation"),
#     detection_metrics(test_results, "Test"),
# ])

# print("FGSM-TRAINED MODEL: CLEAN IMAGE PERFORMANCE")

# display(clean_metrics.style.format({
#     "loss": "{:.4f}",
#     "accuracy": "{:.2%}",
#     "TPR / recall": "{:.2%}",
#     "FPR": "{:.2%}",
#     "precision": "{:.2%}",
# }))

# print(f"Test accuracy: {test_results['accuracy']:.2%}")

In [ ]:
# from torch.utils.data import TensorDataset, DataLoader

# epsilon = 0.03

# fgsm_adv_images_list = []
# fgsm_labels_list = []

# # Generate FGSM adversarial images from the test set
# for images, labels in testloader:
#     images = images.to(device)
#     labels = labels.to(device)

#     adv_images = fgsm_attack(
#         fgsm_adv_model,
#         images,
#         labels,
#         epsilon
#     )

#     fgsm_adv_images_list.append(adv_images.cpu())
#     fgsm_labels_list.append(labels.cpu())

# # Combine all adversarial images and labels
# fgsm_adv_images = torch.cat(fgsm_adv_images_list)
# fgsm_adv_labels = torch.cat(fgsm_labels_list)

# # Create a DataLoader for the adversarial test set
# fgsm_test_dataset = TensorDataset(fgsm_adv_images, fgsm_adv_labels)
# fgsm_testloader = DataLoader(
#     fgsm_test_dataset,
#     batch_size=testloader.batch_size,
#     shuffle=False
# )

# # Evaluate the FGSM-trained model on adversarial test images
# fgsm_test_results = evaluate_model(
#     fgsm_adv_model,
#     fgsm_testloader,
#     criterion,
#     device
# )

# # Display adversarial test metrics
# fgsm_metrics = pd.DataFrame([
#     detection_metrics(fgsm_test_results, "FGSM-attacked Test")
# ])

# print("FGSM-TRAINED MODEL: ADVERSARIAL PERFORMANCE")

# display(fgsm_metrics.style.format({
#     "loss": "{:.4f}",
#     "accuracy": "{:.2%}",
#     "TPR / recall": "{:.2%}",
#     "FPR": "{:.2%}",
#     "precision": "{:.2%}",
# }))

# print(f"Accuracy under FGSM attack: {fgsm_test_results['accuracy']:.2%}")
# print(f"TPR under FGSM attack: {fgsm_metrics.iloc[0]['TPR / recall']:.2%}")
# print(f"FPR under FGSM attack: {fgsm_metrics.iloc[0]['FPR']:.2%}")

In [ ]:
# # How often the attack makes a correctly detected watermark go undetected
# import numpy as np

# clean_targets = np.asarray(test_results["targets"])
# clean_preds = np.asarray(test_results["preds"])
# adv_preds = np.asarray(fgsm_test_results["preds"])

# # Select watermarked images correctly detected before the attack
# correct_watermarked = (clean_targets == 1) & (clean_preds == 1)

# # Count how many are classified as unwatermarked after the attack
# successful_attacks = correct_watermarked & (adv_preds == 0)

# if correct_watermarked.sum() > 0:
#     attack_success_rate = successful_attacks.sum() / correct_watermarked.sum()
# else:
#     attack_success_rate = float("nan")

# print(f"FGSM Attack Success Rate: {attack_success_rate:.2%}")

### Adversarial training (PGD)
LLM disclosure:
Code below is written with assistance from ChatGPT with human review to ensure code is correct.

In [ ]:
# # pgd attack
# def train_pgd_model_cifar10(model, trainloader, valloader, optimizer, criterion,
#                               max_epochs, device, epsilon, alpha, num_iter, tolerance=1e-6):
#     history = []
#     prev_loss = float("inf")
#     model.to(device)
#     use_amp = device.type == "cuda"
#     scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

#     for epoch in range(max_epochs):
#         model.train()
#         running_loss, correct, total = 0.0, 0, 0
#         last_batch_loss = None

#         for images, targets in tqdm(trainloader, desc=f"Epoch {epoch+1}/{max_epochs}", leave=False):
#             images = images.to(device, non_blocking=True)
#             targets = targets.to(device, non_blocking=True)

#             # Generate adversarial images
#             model.eval()
#             adv_images = pgd_attack(model, images, targets, epsilon, alpha, num_iter)
    
#             # Combine original and adversarial images
#             combined_images = torch.cat([images, adv_images], dim=0)
#             combined_targets = torch.cat([targets, targets], dim=0)

#             model.train()
#             optimizer.zero_grad(set_to_none=True)

#             with torch.autocast("cuda", enabled=use_amp):
#                 logits = model(combined_images)
#                 loss = criterion(logits, combined_targets)

#             scaler.scale(loss).backward()
#             scaler.step(optimizer)
#             scaler.update()

#             last_batch_loss = loss.item()
#             running_loss += last_batch_loss * combined_targets.size(0)
#             correct += (logits.argmax(dim=1) == combined_targets).sum().item()
#             total += combined_targets.size(0)

#         if last_batch_loss is None:
#             raise ValueError("Training dataloader is empty.")

#         val_results = evaluate_model(model, valloader, criterion, device)
#         row = {
#             "epoch": epoch + 1,
#             "train_loss": running_loss / total,
#             "last_batch_loss": last_batch_loss,
#             "train_accuracy": correct / total,
#             "val_loss": val_results["loss"],
#             "val_accuracy": val_results["accuracy"],
#         }
#         history.append(row)
#         print(f"Epoch {epoch+1:02d} | last batch loss {last_batch_loss:.6f} | "
#               f"train loss {row['train_loss']:.4f} | "
#               f"train acc {row['train_accuracy']:.2%} | "
#               f"val loss {row['val_loss']:.4f} | "
#               f"val acc {row['val_accuracy']:.2%}")

#         if epoch > 0:
#             loss_diff = prev_loss - last_batch_loss
#             if loss_diff < tolerance:
#                 print(f"Early stopping at epoch {epoch}")
#                 break
#         prev_loss = last_batch_loss

#     return model, pd.DataFrame(history)

In [ ]:
# # new ResNet50 for PGD adversarial training
# pgd_adv_model = models.resnet50(weights=WEIGHTS)

# # Replace classification head
# pgd_adv_model.fc = nn.Linear(pgd_adv_model.fc.in_features, 2)

# pgd_adv_model = pgd_adv_model.to(device)

# # Create a new optimizer
# pgd_adv_optimizer = torch.optim.SGD(
#     pgd_adv_model.parameters(),
#     lr=LEARNING_RATE,
#     momentum=MOMENTUM
# )

# print("PGD Model:", type(pgd_adv_model).__name__)
# print("Classification head:", pgd_adv_model.fc)

In [ ]:
# epsilon = [0.005, 0.01, 0.02, 0.03]
# alpha = [0.001, 0.002, 0.004, 0.006]
# num_iter = 10
# tolerance = 1e-6
# all_pgd_results = []

# for i in range(len(epsilon)):
#     # new ResNet50 for PGD adversarial training
#     pgd_adv_model = models.resnet50(weights=WEIGHTS)

#     # Replace classification head
#     pgd_adv_model.fc = nn.Linear(pgd_adv_model.fc.in_features, 2)

#     pgd_adv_model = pgd_adv_model.to(device)

#     # Create a new optimizer
#     pgd_adv_optimizer = torch.optim.SGD(
#         pgd_adv_model.parameters(),
#         lr=LEARNING_RATE,
#         momentum=MOMENTUM
#     )

#     print("PGD Model:", type(pgd_adv_model).__name__)
#     print("Classification head:", pgd_adv_model.fc)

#     pgd_model, pgd_history = train_pgd_model_cifar10(
#         model=pgd_adv_model,
#         trainloader=trainloader,
#         valloader=valloader,
#         optimizer=pgd_adv_optimizer,
#         criterion=criterion,
#         max_epochs=10,
#         device=device,
#         epsilon=epsilon[i],
#         alpha=alpha[i],
#         num_iter=num_iter,
#         tolerance=tolerance
#     )

#     # show epoch history
#     print(display(pgd_history))


#     # Evaluate the PGD-trained model on clean images
#     train_results = evaluate_model(pgd_model, trainloader, criterion, device)
#     val_results = evaluate_model(pgd_model, valloader, criterion, device)
#     test_results = evaluate_model(pgd_model, testloader, criterion, device)

#     # Create a metrics table
#     clean_metrics = pd.DataFrame([
#         detection_metrics(train_results, "Train"),
#         detection_metrics(val_results, "Validation"),
#         detection_metrics(test_results, "Test"),
#     ])

#     print("PGD-TRAINED MODEL: CLEAN IMAGE PERFORMANCE (using pgd_model)")

#     display(clean_metrics.style.format({
#         "loss": "{:.4f}",
#         "accuracy": "{:.2%}",
#         "TPR / recall": "{:.2%}",
#         "FPR": "{:.2%}",
#         "precision": "{:.2%}",
#     }))

#     print(f"Test accuracy: {test_results['accuracy']:.2%}")


#     from torch.utils.data import TensorDataset, DataLoader

#     pgd_adv_images_list = []
#     pgd_labels_list = []

#     # Generate PGD adversarial images from the test set
#     for images, labels in testloader:
#         images = images.to(device)
#         labels = labels.to(device)

#         adv_images = pgd_attack(
#             pgd_model,
#             images,
#             labels,
#             alpha=alpha[i],
#             iterations=10,
#             epsilon=epsilon[i]
#         )

#         pgd_adv_images_list.append(adv_images.cpu())
#         pgd_labels_list.append(labels.cpu())

#     # Combine all adversarial images and labels
#     pgd_adv_images = torch.cat(pgd_adv_images_list)
#     pgd_adv_labels = torch.cat(pgd_labels_list)

#     # Create a DataLoader for the adversarial test set
#     pgd_test_dataset = TensorDataset(pgd_adv_images, pgd_adv_labels)
#     pgd_testloader = DataLoader(
#         pgd_test_dataset,
#         batch_size=testloader.batch_size,
#         shuffle=False
#     )

#     # Evaluate the PGD-trained model on adversarial test images
#     pgd_test_results = evaluate_model(
#         pgd_model,
#         pgd_testloader,
#         criterion,
#         device
#     )

#     # Display adversarial test metrics
#     pgd_metrics = pd.DataFrame([
#         detection_metrics(pgd_test_results, "PGD-attacked Test")
#     ])

#     print("PGD-TRAINED MODEL: ADVERSARIAL PERFORMANCE")

#     display(pgd_metrics.style.format({
#         "loss": "{:.4f}",
#         "accuracy": "{:.2%}",
#         "TPR / recall": "{:.2%}",
#         "FPR": "{:.2%}",
#         "precision": "{:.2%}",
#     }))

#     print(f"Accuracy under PGD attack: {pgd_test_results['accuracy']:.2%}")
#     print(f"TPR under PGD attack: {pgd_metrics.iloc[0]['TPR / recall']:.2%}")
#     print(f"FPR under PGD attack: {pgd_metrics.iloc[0]['FPR']:.2%}")


#     # How often the attack makes a correctly detected watermark go undetected
#     import numpy as np

#     clean_targets = np.asarray(test_results["targets"])
#     clean_preds = np.asarray(test_results["preds"])
#     adv_preds = np.asarray(pgd_test_results["preds"])

#     # Select watermarked images correctly detected before the attack
#     correct_watermarked = (clean_targets == 1) & (clean_preds == 1)

#     # Count how many are classified as unwatermarked after the attack
#     successful_attacks = correct_watermarked & (adv_preds == 0)

#     if correct_watermarked.sum() > 0:
#         attack_success_rate = successful_attacks.sum() / correct_watermarked.sum()
#     else:
#         attack_success_rate = float("nan")

#     print(f"PGD Attack Success Rate: {attack_success_rate:.2%}")

#     all_pgd_results.append({
#         "epsilon": epsilon[i],
#         "alpha": alpha[i],
#         "clean_test_accuracy": test_results['accuracy'],
#         "pgd_test_accuracy": pgd_test_results['accuracy'],
#         "attack_success_rate": attack_success_rate
#     })

#     print("####################### NEW SET OF EPSILON AND ALPHA #############################################")
    

In [ ]:
# pgd_results = pd.DataFrame(all_pgd_results)
# display(pgd_results)

In [ ]:
# display(pgd_history)

In [ ]:
# # Evaluate the PGD-trained model on clean images
# train_results = evaluate_model(pgd_model, trainloader, criterion, device)
# val_results = evaluate_model(pgd_model, valloader, criterion, device)
# test_results = evaluate_model(pgd_model, testloader, criterion, device)

# # Create a metrics table
# clean_metrics = pd.DataFrame([
#     detection_metrics(train_results, "Train"),
#     detection_metrics(val_results, "Validation"),
#     detection_metrics(test_results, "Test"),
# ])

# print("PGD-TRAINED MODEL: CLEAN IMAGE PERFORMANCE (using pgd_model)")

# display(clean_metrics.style.format({
#     "loss": "{:.4f}",
#     "accuracy": "{:.2%}",
#     "TPR / recall": "{:.2%}",
#     "FPR": "{:.2%}",
#     "precision": "{:.2%}",
# }))

# print(f"Test accuracy: {test_results['accuracy']:.2%}")

In [ ]:
# from torch.utils.data import TensorDataset, DataLoader

# epsilon = 0.03

# pgd_adv_images_list = []
# pgd_labels_list = []

# # Generate PGD adversarial images from the test set
# for images, labels in testloader:
#     images = images.to(device)
#     labels = labels.to(device)

#     adv_images = pgd_attack(
#         pgd_model,
#         images,
#         labels,
#         alpha=0.01,
#         iterations=10,
#         epsilon=epsilon
#     )

#     pgd_adv_images_list.append(adv_images.cpu())
#     pgd_labels_list.append(labels.cpu())

# # Combine all adversarial images and labels
# pgd_adv_images = torch.cat(pgd_adv_images_list)
# pgd_adv_labels = torch.cat(pgd_labels_list)

# # Create a DataLoader for the adversarial test set
# pgd_test_dataset = TensorDataset(pgd_adv_images, pgd_adv_labels)
# pgd_testloader = DataLoader(
#     pgd_test_dataset,
#     batch_size=testloader.batch_size,
#     shuffle=False
# )

# # Evaluate the PGD-trained model on adversarial test images
# pgd_test_results = evaluate_model(
#     pgd_model,
#     pgd_testloader,
#     criterion,
#     device
# )

# # Display adversarial test metrics
# pgd_metrics = pd.DataFrame([
#     detection_metrics(pgd_test_results, "PGD-attacked Test")
# ])

# print("PGD-TRAINED MODEL: ADVERSARIAL PERFORMANCE")

# display(pgd_metrics.style.format({
#     "loss": "{:.4f}",
#     "accuracy": "{:.2%}",
#     "TPR / recall": "{:.2%}",
#     "FPR": "{:.2%}",
#     "precision": "{:.2%}",
# }))

# print(f"Accuracy under PGD attack: {pgd_test_results['accuracy']:.2%}")
# print(f"TPR under PGD attack: {pgd_metrics.iloc[0]['TPR / recall']:.2%}")
# print(f"FPR under PGD attack: {pgd_metrics.iloc[0]['FPR']:.2%}")

In [ ]:
# # How often the attack makes a correctly detected watermark go undetected
# import numpy as np

# clean_targets = np.asarray(test_results["targets"])
# clean_preds = np.asarray(test_results["preds"])
# adv_preds = np.asarray(pgd_test_results["preds"])

# # Select watermarked images correctly detected before the attack
# correct_watermarked = (clean_targets == 1) & (clean_preds == 1)

# # Count how many are classified as unwatermarked after the attack
# successful_attacks = correct_watermarked & (adv_preds == 0)

# if correct_watermarked.sum() > 0:
#     attack_success_rate = successful_attacks.sum() / correct_watermarked.sum()
# else:
#     attack_success_rate = float("nan")

# print(f"PGD Attack Success Rate: {attack_success_rate:.2%}")

In [ ]:
# all_preds = []
# all_labels = []

# pgd_model.eval()

# with torch.no_grad():
#     for images, labels in valloader:
#         images = images.to(device)

#         outputs = pgd_model(images)
#         preds = outputs.argmax(dim=1)

#         all_preds.extend(preds.cpu().numpy())
#         all_labels.extend(labels.numpy())

# print("Predicted as unwatermarked:", np.sum(np.array(all_preds) == 0))
# print("Predicted as watermarked:", np.sum(np.array(all_preds) == 1))